Week 6 -Spark Architecture and Data Processing
Name: Unnati Agarwal Internship: Celebal Technologies

In [2]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week6 Spark Assignment") \
    .getOrCreate()

In [4]:
from google.colab import files

uploaded = files.upload()

Saving sample_sales.csv to sample_sales.csv


Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

Ans - In a Spark application, the Driver acts as the main controller. It creates the Spark session, divides the work into smaller tasks, and coordinates their execution.

The Cluster Manager is responsible for managing the available resources in the cluster. It allocates CPU and memory to different Spark applications and starts the executors.

The Executors are the worker processes that actually perform the computations. They process the assigned tasks, store data in memory when required, and send the results back to the Driver.

Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

Ans- Spark does not execute transformations immediately. Instead, it records all the operations and waits until an action like show() or count() is called. This approach is known as Lazy Evaluation.

By waiting until the end, Spark can optimize the execution plan, remove unnecessary operations, and combine multiple transformations into a more efficient workflow. This reduces processing time and improves the performance of large-scale data processing.

Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [5]:
df = spark.read.csv(
    "sample_sales.csv",
    header=True,
    inferSchema=True
)

df.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   status|age|subscription|city|     email|username|price|store_id|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    102|      2024-01-06|  East|       Furniture|      320.0|     NULL| 34|       Basic|  NY|b@mail.com|     bob|320.0|    S102|2024-01-06 11:20:00|
|    103|      2024-01-07|  West|     Electronics|      610.0|Completed| 29|     Premium|  SF|      

Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

Ans- CSV is a row-based file format where data is stored one row after another. It is simple and easy to read but usually takes more storage space and is slower to process.

Parquet is a columnar file format where values from the same column are stored together. Since Spark often reads only a few columns at a time, Parquet reduces the amount of data that needs to be loaded. It also supports compression, making it faster and more storage-efficient than CSV.

Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [7]:
df.select("user_id", "price") \
  .filter(df.product_category == "Electronics") \
  .show()

+-------+-----+
|user_id|price|
+-------+-----+
|    101|450.0|
|    101|450.0|
|    103|610.0|
|    105| NULL|
|    108|530.0|
|    111|480.0|
|    112|410.0|
+-------+-----+



Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [8]:
from pyspark.sql.functions import col

df = (
    df.withColumnRenamed("status", "order_status")
      .withColumn("price", col("price").cast("double"))
)

df.show()

+-------+----------------+------+----------------+-----------+------------+---+------------+----+----------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|order_status|age|subscription|city|     email|username|price|store_id|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------------+---+------------+----+----------+--------+-----+--------+-------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|   Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    101|      2024-01-05|  West|     Electronics|      450.0|   Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    102|      2024-01-06|  East|       Furniture|      320.0|        NULL| 34|       Basic|  NY|b@mail.com|     bob|320.0|    S102|2024-01-06 11:20:00|
|    103|      2024-01-07|  West|     Electronics|      610.0|   Completed| 29|   

Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Ans- Spark keeps track of every transformation applied to the data using a Lineage Graph, also called a DAG (Directed Acyclic Graph).

If an executor or worker node fails, Spark does not need to reload the entire dataset. Instead, it uses the lineage graph to identify the missing data and recomputes only the lost partitions. This makes Spark reliable and fault tolerant without creating unnecessary copies of data.

Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [17]:
df.filter(
    (df.order_status == "Completed") &
    (df.sale_amount > 1000)
).show()

+-------+----------------+------+----------------+-----------+------------+---+------------+----+-----+--------+-----+--------+-------------+-----------+
|user_id|transaction_date|region|product_category|sale_amount|order_status|age|subscription|city|email|username|price|store_id|raw_timestamp|final_price|
+-------+----------------+------+----------------+-----------+------------+---+------------+----+-----+--------+-----+--------+-------------+-----------+
+-------+----------------+------+----------------+-----------+------------+---+------------+----+-----+--------+-----+--------+-------------+-----------+



Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Ans - Predicate Pushdown is an optimization feature used with Parquet files. When a filter condition is applied, Spark sends that condition directly to the Parquet file reader.

As a result, only the required rows are read into memory instead of loading the entire dataset. This reduces memory usage, minimizes disk I/O, and improves the overall performance of the application.

Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [11]:
from pyspark.sql.functions import col

df = df.withColumn(
    "final_price",
    col("price") * 1.18
)

df.show()

+-------+----------------+------+----------------+-----------+------------+---+------------+----+----------+--------+-----+--------+-------------------+------------------+
|user_id|transaction_date|region|product_category|sale_amount|order_status|age|subscription|city|     email|username|price|store_id|      raw_timestamp|       final_price|
+-------+----------------+------+----------------+-----------+------------+---+------------+----+----------+--------+-----+--------+-------------------+------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|   Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|             531.0|
|    101|      2024-01-05|  West|     Electronics|      450.0|   Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|             531.0|
|    102|      2024-01-06|  East|       Furniture|      320.0|        NULL| 34|       Basic|  NY|b@mail.com|     bob|320.0|    S102|2024-01-

Q11: What is the difference between Transformations and Actions? Provide two examples of each.

Ans -

Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".


In [20]:

# Read the CSV file
df = spark.read.csv(
    "sample_sales.csv",
    header=True,
    inferSchema=True
)

# Save it as Parquet (to simulate the assignment input)
df.write.mode("overwrite").parquet("input_parquet")

# Read the Parquet file
parquet_df = spark.read.parquet("input_parquet")

# Filter out rows where user_id is null
clean_df = parquet_df.filter(parquet_df.user_id.isNotNull())

# Save the cleaned data as CSV
clean_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output_csv")

# Display the cleaned data
clean_df.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   status|age|subscription|city|     email|username|price|store_id|      raw_timestamp|
+-------+----------------+------+----------------+-----------+---------+---+------------+----+----------+--------+-----+--------+-------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    101|      2024-01-05|  West|     Electronics|      450.0|Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|
|    102|      2024-01-06|  East|       Furniture|      320.0|     NULL| 34|       Basic|  NY|b@mail.com|     bob|320.0|    S102|2024-01-06 11:20:00|
|    103|      2024-01-07|  West|     Electronics|      610.0|Completed| 29|     Premium|  SF|      

Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

Ans - In Client Mode, the Driver program runs on the user's local machine. This mode is mainly used for development, testing, and debugging because the user can directly monitor the application.

In Cluster Mode, the Driver runs inside the cluster instead of the local machine. This makes the application more reliable because it continues running even if the client disconnects. Cluster Mode is commonly used for production environments.

Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High.

In [13]:
df.filter(
    (df.region == "North") |
    (df.subscription == "Premium")
).show()

+-------+----------------+------+----------------+-----------+------------+---+------------+----+----------+--------+-----+--------+-------------------+------------------+
|user_id|transaction_date|region|product_category|sale_amount|order_status|age|subscription|city|     email|username|price|store_id|      raw_timestamp|       final_price|
+-------+----------------+------+----------------+-----------+------------+---+------------+----+----------+--------+-----+--------+-------------------+------------------+
|    101|      2024-01-05|  West|     Electronics|      450.0|   Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|             531.0|
|    101|      2024-01-05|  West|     Electronics|      450.0|   Completed| 25|     Premium|  SF|a@mail.com|   alice|450.0|    S101|2024-01-05 10:15:00|             531.0|
|    103|      2024-01-07|  West|     Electronics|      610.0|   Completed| 29|     Premium|  SF|      NULL|   carol|610.0|    S101|2024-01-

Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

Ans-The .show(5) function displays only the first five rows of a dataset, making it a safe way to quickly inspect the data without consuming much memory.

On the other hand, .collect() retrieves the entire dataset and stores it in the Driver's memory. For very large datasets, this can lead to excessive memory usage or even crash the application. Therefore, using .show(5) is a much safer option when working with big data.